# Associate Catalog Objects

Propagates the historical orbital catalog to each event time, applies frozen residual gates, and separates strict, supported, and unresolved associations.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


In [ ]:
import importlib.util
import subprocess
import sys

for package in ['sgp4', 'astropy']:
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])

print('Environment ready. No Hardtarget, GMF, or Swerling work will run.')


In [ ]:
from pathlib import Path
import gzip
import hashlib
import json
import math
import os
import time

import numpy as np
import pandas as pd

BASE = Path(os.environ.get('DEBRIS_BASE', str(PROJECT_ROOT)))
ORIGINAL_ROOT = BASE / 'phase0_outputs/catalog_matching/ca_cfar_1p75_24h'
CONSOLIDATION_ROOT = BASE / 'phase0_outputs/catalog_matching/ca_cfar_1p75_24h_consolidated'
OUT = BASE / 'phase0_outputs/catalog_matching/ca_cfar_1p75_24h_final_audited'
CHECKPOINTS = OUT / 'checkpoints'
ACTUAL_CHECKPOINTS = CHECKPOINTS / 'actual'
SHIFT_CHECKPOINTS = CHECKPOINTS / 'time_shift_controls'
for folder in [OUT, CHECKPOINTS, ACTUAL_CHECKPOINTS, SHIFT_CHECKPOINTS]:
    folder.mkdir(parents=True, exist_ok=True)

SELECTED_PATH = ORIGINAL_ROOT / 'selected_object_detections.csv'
ORIGINAL_ASSOC_PATH = ORIGINAL_ROOT / 'catalog_association_summary.csv'
CONSOLIDATED_PATH = CONSOLIDATION_ROOT / 'consolidated_object_detections.csv'
MEMBERSHIP_PATH = CONSOLIDATION_ROOT / 'fragment_membership.csv'
LINKS_PATH = CONSOLIDATION_ROOT / 'accepted_fragment_links.csv'
HISTORY_PATH = (
    BASE / 'phase0_outputs/catalog_matching/historical_catalog'
    / 'gp_history_20180103_20180106.json.gz'
)

required = [
    SELECTED_PATH, ORIGINAL_ASSOC_PATH, CONSOLIDATED_PATH,
    MEMBERSHIP_PATH, LINKS_PATH, HISTORY_PATH,
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError('Missing required input(s):\n' + '\n'.join(missing))

                                               
EXPECTED_CONFIG_ID = 'ca_1.75'
EXPECTED_METHOD = 'CA-CFAR'
EXPECTED_THRESHOLD = 1.75
EXPECTED_TRACK_RULE = 'short_0p8'
PAPER_RANGE_FLOOR_KM = 1.0
PAPER_RATE_FLOOR_KM_S = 0.020
PRIMARY_RANGE_FRACTION = 0.001
PRIMARY_RATE_FRACTION = 0.01
BEAM_LIMIT_DEG = 5.0

                                                        
EMPIRICAL_CHI2_99 = 9.21034037197618                         
EMPIRICAL_ABS_RANGE_CAP_KM = 3.0
EMPIRICAL_ABS_RATE_CAP_KM_S = 0.060
EMPIRICAL_MIN_D2_MARGIN = 4.0
EMPIRICAL_RANGE_SIGMA_FLOOR_KM = 0.15
EMPIRICAL_RATE_SIGMA_FLOOR_KM_S = 0.005
QUALITY_MIN_DURATION_SEC = 5.0
QUALITY_MIN_SCORE = 30.0
MAX_CONTROL_FDR = 0.10

                                                                                   
TIME_SHIFT_CONTROLS_SEC = [-1800, -900, 900, 1800]
TOP_CANDIDATES_TO_SAVE = 50
FORCE_RECOMPUTE = False

selected = pd.read_csv(SELECTED_PATH)
original_assoc = pd.read_csv(ORIGINAL_ASSOC_PATH)
consolidated = pd.read_csv(CONSOLIDATED_PATH)
membership = pd.read_csv(MEMBERSHIP_PATH)
accepted_links = pd.read_csv(LINKS_PATH)

for column in ['config_id', 'method', 'threshold', 'track_rule']:
    if column not in selected:
        raise KeyError(f'Selected-event table is missing {column!r}')
assert set(selected['config_id'].astype(str)) == {EXPECTED_CONFIG_ID}
assert set(selected['method'].astype(str)) == {EXPECTED_METHOD}
assert np.allclose(pd.to_numeric(selected['threshold']), EXPECTED_THRESHOLD)
assert set(selected['track_rule'].astype(str)) == {EXPECTED_TRACK_RULE}

required_consolidated = {
    'consolidated_detection_id', 'fit_center_time_unix', 'fitted_range_km',
    'fitted_range_rate_km_s', 'start_time_unix', 'end_time_unix',
    'duration_sec', 'number_fragments', 'member_detection_ids',
}
if not required_consolidated.issubset(consolidated.columns):
    raise KeyError('Consolidated-event table has an unexpected schema.')

selected_ids = set(selected['object_detection_id'].astype(str))
member_ids = membership['object_detection_id'].astype(str)
assert len(selected) == 164, f'Expected 164 frozen fragments, found {len(selected)}'
assert len(consolidated) == 155, f'Expected 155 consolidated events, found {len(consolidated)}'
assert len(membership) == len(selected)
assert member_ids.nunique() == len(selected)
assert set(member_ids) == selected_ids
assert membership['consolidated_detection_id'].nunique() == len(consolidated)

                                                                     
member_detail = membership.merge(
    selected,
    on='object_detection_id',
    how='left',
    validate='one_to_one',
)

def join_unique(values):
    return ';'.join(sorted({str(value) for value in values}))

group_meta = member_detail.groupby('consolidated_detection_id').agg(
    hour_key=('hour_key', join_unique),
    number_unique_times=('number_unique_times', 'sum'),
    peak_detector_score=('peak_detector_score', 'max'),
    acceleration_km_s2=('acceleration_km_s2', 'median'),
    member_trajectory_rmse_km=('trajectory_rmse_km', 'median'),
    beam_azimuth_deg=('beam_azimuth_deg', 'median'),
    beam_elevation_deg=('beam_elevation_deg', 'median'),
).reset_index()

objects = consolidated.merge(
    group_meta, on='consolidated_detection_id', how='left', validate='one_to_one'
).rename(columns={
    'consolidated_detection_id': 'event_id',
    'fit_center_time_unix': 'reference_time_unix',
    'fitted_range_km': 'observed_range_km',
    'fitted_range_rate_km_s': 'observed_range_rate_km_s',
})
objects['trajectory_rmse_km'] = pd.to_numeric(
    objects.get('trajectory_rmse_km'), errors='coerce'
).fillna(pd.to_numeric(objects['member_trajectory_rmse_km'], errors='coerce'))
objects = objects.sort_values('reference_time_unix').reset_index(drop=True)

CONFIG_PAYLOAD = {
    'version': 'final_audited_catalog_v1',
    'detector': [EXPECTED_CONFIG_ID, EXPECTED_THRESHOLD, EXPECTED_TRACK_RULE],
    'primary_gates': [PAPER_RANGE_FLOOR_KM, PAPER_RATE_FLOOR_KM_S,
                      PRIMARY_RANGE_FRACTION, PRIMARY_RATE_FRACTION,
                      BEAM_LIMIT_DEG],
    'empirical': [EMPIRICAL_CHI2_99, EMPIRICAL_ABS_RANGE_CAP_KM,
                  EMPIRICAL_ABS_RATE_CAP_KM_S, EMPIRICAL_MIN_D2_MARGIN,
                  EMPIRICAL_RANGE_SIGMA_FLOOR_KM,
                  EMPIRICAL_RATE_SIGMA_FLOOR_KM_S],
    'time_shifts_sec': TIME_SHIFT_CONTROLS_SEC,
}
CONFIG_HASH = hashlib.sha256(
    json.dumps(CONFIG_PAYLOAD, sort_keys=True).encode('utf-8')
).hexdigest()[:16]

print('Frozen detector fragments:', len(selected))
print('Accepted fragment links:', len(accepted_links))
print('Consolidated radar events:', len(objects))
print('Multi-fragment events:', int((objects.number_fragments > 1).sum()))
print('Configuration hash:', CONFIG_HASH)
print('Output folder:', OUT)


## 1. Full-catalog SGP4 propagation and physical geometry


In [ ]:
import warnings

import astropy.units as u
from astropy.coordinates import (
    CartesianDifferential,
    CartesianRepresentation,
    EarthLocation,
    ITRS,
    TEME,
)
from astropy.time import Time
from astropy.utils import iers
from sgp4.api import Satrec, SatrecArray

iers.conf.auto_download = False
iers.conf.auto_max_age = None

RADAR_LAT_DEG = 69.0 + 35.0 / 60.0 + 11.0 / 3600.0
RADAR_LON_DEG = 19.0 + 13.0 / 60.0 + 38.0 / 3600.0
RADAR_ALT_M = 86.0

station = EarthLocation.from_geodetic(
    lon=RADAR_LON_DEG * u.deg,
    lat=RADAR_LAT_DEG * u.deg,
    height=RADAR_ALT_M * u.m,
)

with gzip.open(HISTORY_PATH, 'rt', encoding='utf-8') as handle:
    history = pd.DataFrame(json.load(handle))

required_history = ['NORAD_CAT_ID', 'EPOCH', 'TLE_LINE1', 'TLE_LINE2']
missing_history = [column for column in required_history if column not in history]
if missing_history:
    raise KeyError('Historical cache is missing: ' + ', '.join(missing_history))

history['NORAD_CAT_ID'] = pd.to_numeric(
    history['NORAD_CAT_ID'], errors='coerce'
).astype('Int64')
history['EPOCH_UTC'] = pd.to_datetime(history['EPOCH'], utc=True, errors='coerce')
history = history.dropna(
    subset=['NORAD_CAT_ID', 'EPOCH_UTC', 'TLE_LINE1', 'TLE_LINE2']
).sort_values(['NORAD_CAT_ID', 'EPOCH_UTC']).reset_index(drop=True)

SATREC_CACHE = {}


def teme_to_radar_geometry(r_km, v_km_s, obstime, beam_az_deg, beam_el_deg):
    r_km = np.asarray(r_km, dtype=float)
    v_km_s = np.asarray(v_km_s, dtype=float)
    if r_km.ndim == 1:
        r_km = r_km[None, :]
        v_km_s = v_km_s[None, :]

    representation = CartesianRepresentation(
        x=r_km[:, 0] * u.km,
        y=r_km[:, 1] * u.km,
        z=r_km[:, 2] * u.km,
        differentials=CartesianDifferential(
            d_x=v_km_s[:, 0] * u.km / u.s,
            d_y=v_km_s[:, 1] * u.km / u.s,
            d_z=v_km_s[:, 2] * u.km / u.s,
        ),
    )
    teme = TEME(representation, obstime=obstime)
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        itrs = teme.transform_to(ITRS(obstime=obstime))

    sat_pos = itrs.cartesian.xyz.to_value(u.km).T
    sat_vel = itrs.cartesian.differentials['s'].d_xyz.to_value(u.km / u.s).T
    station_pos = station.get_itrs(obstime=obstime).cartesian.xyz.to_value(u.km).T
    if station_pos.ndim == 1:
        station_pos = station_pos[None, :]

    rel = sat_pos - station_pos
    slant_range = np.linalg.norm(rel, axis=1)
    radial_rate = np.einsum('ij,ij->i', rel, sat_vel) / slant_range

    lat = np.deg2rad(RADAR_LAT_DEG)
    lon = np.deg2rad(RADAR_LON_DEG)
    x, y, z = rel.T
    east = -np.sin(lon) * x + np.cos(lon) * y
    north = (
        -np.sin(lat) * np.cos(lon) * x
        - np.sin(lat) * np.sin(lon) * y
        + np.cos(lat) * z
    )
    up = (
        np.cos(lat) * np.cos(lon) * x
        + np.cos(lat) * np.sin(lon) * y
        + np.sin(lat) * z
    )
    azimuth = np.rad2deg(np.arctan2(east, north)) % 360.0
    elevation = np.rad2deg(np.arcsin(np.clip(up / slant_range, -1.0, 1.0)))

    az = np.deg2rad(azimuth)
    el = np.deg2rad(elevation)
    beam_az = np.deg2rad(float(beam_az_deg))
    beam_el = np.deg2rad(float(beam_el_deg))
    cosine = np.clip(
        np.cos(el) * np.sin(az) * np.cos(beam_el) * np.sin(beam_az)
        + np.cos(el) * np.cos(az) * np.cos(beam_el) * np.cos(beam_az)
        + np.sin(el) * np.sin(beam_el),
        -1.0,
        1.0,
    )
    beam_separation = np.rad2deg(np.arccos(cosine))
    return {
        'slant_range_km': slant_range,
        'radial_rate_km_s': radial_rate,
        'azimuth_deg': azimuth,
        'elevation_deg': elevation,
        'beam_separation_deg': beam_separation,
    }


def latest_prior_catalog(event_unix):
    event_timestamp = pd.to_datetime(float(event_unix), unit='s', utc=True)
    prior = history[history['EPOCH_UTC'] <= event_timestamp]
    if prior.empty:
        raise RuntimeError('No historical TLE precedes this event.')
    selected_prior = prior.drop_duplicates('NORAD_CAT_ID', keep='last').copy()
    selected_prior['tle_age_hours'] = (
        event_timestamp - selected_prior['EPOCH_UTC']
    ).dt.total_seconds() / 3600.0
    return selected_prior.reset_index(drop=True)


def build_satellite_array(catalog):
    satrecs, kept_rows = [], []
    failures = 0
    for row_index, row in catalog.iterrows():
        key = (str(row['TLE_LINE1']), str(row['TLE_LINE2']))
        try:
            sat = SATREC_CACHE.get(key)
            if sat is None:
                sat = Satrec.twoline2rv(*key)
                SATREC_CACHE[key] = sat
            satrecs.append(sat)
            kept_rows.append(row_index)
        except Exception:
            failures += 1
    if not satrecs:
        raise RuntimeError('No latest-prior TLEs could be parsed.')
    return catalog.loc[kept_rows].reset_index(drop=True), satrecs, failures


def propagate_all(satrecs, obstime, beam_az, beam_el):
    sat_array = SatrecArray(satrecs)
    error, r, v = sat_array.sgp4(
        np.asarray([obstime.jd1], dtype=float),
        np.asarray([obstime.jd2], dtype=float),
    )
    error = np.asarray(error)[:, 0]
    r = np.asarray(r)[:, 0, :]
    v = np.asarray(v)[:, 0, :]
    valid = (
        (error == 0)
        & np.all(np.isfinite(r), axis=1)
        & np.all(np.isfinite(v), axis=1)
    )
    geometry = teme_to_radar_geometry(r[valid], v[valid], obstime, beam_az, beam_el)
    return np.where(valid)[0], geometry


def criterion_status(number):
    if int(number) == 1:
        return 'unique'
    if int(number) > 1:
        return 'ambiguous'
    return 'none'


print(f'Historical records: {len(history):,}')
print(f'Historical catalog objects: {history.NORAD_CAT_ID.nunique():,}')


In [ ]:
def row_signature(row, time_offset_sec=0):
    payload = {
        'event_id': str(row['event_id']),
        'reference_time_unix': float(row['reference_time_unix']),
        'observed_range_km': float(row['observed_range_km']),
        'observed_range_rate_km_s': float(row['observed_range_rate_km_s']),
        'beam_azimuth_deg': float(row['beam_azimuth_deg']),
        'beam_elevation_deg': float(row['beam_elevation_deg']),
        'time_offset_sec': int(time_offset_sec),
    }
    return hashlib.sha256(
        json.dumps(payload, sort_keys=True).encode('utf-8')
    ).hexdigest()[:16]


def atomic_json(payload, path):
    temp = path.with_suffix(path.suffix + '.tmp')
    temp.write_text(json.dumps(payload, indent=2, default=str))
    temp.replace(path)


def atomic_gzip_csv(frame, path):
    temp = path.with_suffix(path.suffix + '.tmp')
    frame.to_csv(temp, index=False, compression='gzip')
    temp.replace(path)


def propagate_event(row, time_offset_sec=0):
    event_id = str(row['event_id'])
    center_unix = float(row['reference_time_unix'])
    propagation_unix = center_unix + float(time_offset_sec)
    obstime = Time(propagation_unix, format='unix', scale='utc')
    observed_range = float(row['observed_range_km'])
    observed_rate = float(row['observed_range_rate_km_s'])
    beam_az = float(row['beam_azimuth_deg'])
    beam_el = float(row['beam_elevation_deg'])

    prior_catalog = latest_prior_catalog(propagation_unix)
    prior_catalog, satrecs, parse_failures = build_satellite_array(prior_catalog)
    valid_indices, geometry = propagate_all(satrecs, obstime, beam_az, beam_el)
    propagated = prior_catalog.iloc[valid_indices].copy().reset_index(drop=True)

    propagated['predicted_slant_range_km'] = geometry['slant_range_km']
    propagated['predicted_radial_rate_km_s'] = geometry['radial_rate_km_s']
    propagated['predicted_azimuth_deg'] = geometry['azimuth_deg']
    propagated['predicted_elevation_deg'] = geometry['elevation_deg']
    propagated['beam_separation_deg'] = geometry['beam_separation_deg']
    propagated['range_residual_km'] = (
        propagated['predicted_slant_range_km'] - observed_range
    )
    propagated['range_rate_residual_km_s'] = (
        propagated['predicted_radial_rate_km_s'] - observed_rate
    )

    range_gate = max(PAPER_RANGE_FLOOR_KM, PRIMARY_RANGE_FRACTION * observed_range)
    rate_gate = max(PAPER_RATE_FLOOR_KM_S, PRIMARY_RATE_FRACTION * abs(observed_rate))
    propagated['primary_range_gate_km'] = range_gate
    propagated['primary_rate_gate_km_s'] = rate_gate
    propagated['within_beam_model'] = propagated['beam_separation_deg'] <= BEAM_LIMIT_DEG
    propagated['above_horizon'] = propagated['predicted_elevation_deg'] > 0.0
    common = propagated['within_beam_model'] & propagated['above_horizon']
    propagated['meets_primary_hybrid_gates'] = (
        (propagated['range_residual_km'].abs() < range_gate)
        & (propagated['range_rate_residual_km_s'].abs() < rate_gate)
        & common
    )
    propagated['meets_paper_fixed_gates'] = (
        (propagated['range_residual_km'].abs() < PAPER_RANGE_FLOOR_KM)
        & (propagated['range_rate_residual_km_s'].abs() < PAPER_RATE_FLOOR_KM_S)
        & common
    )
    propagated['association_metric_primary'] = np.hypot(
        propagated['range_residual_km'] / range_gate,
        propagated['range_rate_residual_km_s'] / rate_gate,
    )
    propagated = propagated.sort_values('association_metric_primary').reset_index(drop=True)
    propagated['rank_within_event'] = np.arange(1, len(propagated) + 1)

    n_primary = int(propagated['meets_primary_hybrid_gates'].sum())
    n_fixed = int(propagated['meets_paper_fixed_gates'].sum())
    strict_candidates = propagated[propagated['meets_primary_hybrid_gates']]
    best = strict_candidates.iloc[0] if n_primary else propagated.iloc[0]
    primary_status = criterion_status(n_primary)
    is_unique = primary_status == 'unique'

    summary = {
        'config_hash': CONFIG_HASH,
        'row_signature': row_signature(row, time_offset_sec),
        'event_id': event_id,
        'member_detection_ids': str(row['member_detection_ids']),
        'number_fragments': int(row['number_fragments']),
        'hour_key': str(row['hour_key']),
        'reference_time_unix': center_unix,
        'reference_time_utc': Time(center_unix, format='unix', scale='utc').utc.isot,
        'propagation_time_offset_sec': int(time_offset_sec),
        'propagation_time_utc': obstime.utc.isot,
        'duration_sec': float(row['duration_sec']),
        'number_unique_times': int(row['number_unique_times']),
        'peak_detector_score': float(row['peak_detector_score']),
        'observed_range_km': observed_range,
        'observed_range_rate_km_s': observed_rate,
        'trajectory_rmse_km': float(row['trajectory_rmse_km']) if pd.notna(row['trajectory_rmse_km']) else None,
        'primary_hybrid_status': primary_status,
        'paper_fixed_status': criterion_status(n_fixed),
        'number_meeting_primary_hybrid': n_primary,
        'number_meeting_fixed': n_fixed,
        'primary_range_gate_km': range_gate,
        'primary_rate_gate_km_s': rate_gate,
        'catalog_objects_with_prior_tle': int(len(prior_catalog)),
        'catalog_objects_propagated': int(len(propagated)),
        'tle_parse_failures': int(parse_failures),
        'matched_NORAD_CAT_ID': int(best['NORAD_CAT_ID']) if is_unique else None,
        'matched_OBJECT_NAME': str(best.get('OBJECT_NAME', '')) if is_unique else None,
        'matched_OBJECT_ID': str(best.get('OBJECT_ID', '')) if is_unique else None,
        'matched_OBJECT_TYPE': str(best.get('OBJECT_TYPE', '')) if is_unique else None,
        'matched_RCS_SIZE': str(best.get('RCS_SIZE', '')) if is_unique else None,
        'diagnostic_best_NORAD_CAT_ID': int(best['NORAD_CAT_ID']),
        'diagnostic_best_OBJECT_NAME': str(best.get('OBJECT_NAME', '')),
        'TLE_EPOCH_UTC': str(best.get('EPOCH_UTC', '')),
        'tle_age_hours': float(best.get('tle_age_hours', np.nan)),
        'predicted_slant_range_km': float(best['predicted_slant_range_km']),
        'range_residual_km': float(best['range_residual_km']),
        'predicted_radial_rate_km_s': float(best['predicted_radial_rate_km_s']),
        'range_rate_residual_km_s': float(best['range_rate_residual_km_s']),
        'normalized_primary_range_residual': float(best['range_residual_km'] / range_gate),
        'normalized_primary_rate_residual': float(best['range_rate_residual_km_s'] / rate_gate),
        'predicted_azimuth_deg': float(best['predicted_azimuth_deg']),
        'predicted_elevation_deg': float(best['predicted_elevation_deg']),
        'beam_separation_deg': float(best['beam_separation_deg']),
        'best_candidate_metric_primary': float(best['association_metric_primary']),
    }

    save_rows = propagated[
        (propagated['rank_within_event'] <= TOP_CANDIDATES_TO_SAVE)
        | propagated['meets_primary_hybrid_gates']
        | propagated['meets_paper_fixed_gates']
    ].copy()
    save_rows.insert(0, 'event_id', event_id)
    save_rows.insert(1, 'reference_time_unix', center_unix)
    save_rows.insert(2, 'propagation_time_offset_sec', int(time_offset_sec))
    save_rows.insert(3, 'observed_range_km', observed_range)
    save_rows.insert(4, 'observed_range_rate_km_s', observed_rate)
    return summary, save_rows


def process_campaign(time_offset_sec, checkpoint_folder, label):
    checkpoint_folder.mkdir(parents=True, exist_ok=True)
    summaries, candidate_frames = [], []
    compute_seconds = []
    resumed = 0
    start_clock = time.perf_counter()
    for index, (_, row) in enumerate(objects.iterrows(), start=1):
        event_id = str(row['event_id'])
        summary_path = checkpoint_folder / f'{event_id}_summary.json'
        candidates_path = checkpoint_folder / f'{event_id}_candidates.csv.gz'
        reusable = False
        if not FORCE_RECOMPUTE and summary_path.exists() and candidates_path.exists():
            try:
                cached = json.loads(summary_path.read_text())
                reusable = (
                    cached.get('config_hash') == CONFIG_HASH
                    and cached.get('row_signature') == row_signature(row, time_offset_sec)
                )
            except Exception:
                reusable = False

        if reusable:
            summary = cached
            candidates = pd.read_csv(candidates_path)
            resumed += 1
            mode = 'checkpoint'
        else:
            event_clock = time.perf_counter()
            summary, candidates = propagate_event(row, time_offset_sec)
            atomic_gzip_csv(candidates, candidates_path)
            atomic_json(summary, summary_path)
            event_seconds = time.perf_counter() - event_clock
            compute_seconds.append(event_seconds)
            mode = f'computed {event_seconds:.1f}s'

        summaries.append(summary)
        candidate_frames.append(candidates)
        if compute_seconds:
            eta = np.median(compute_seconds[-10:]) * (len(objects) - index) / 60.0
            eta_text = f'; ETA≈{eta:.1f} min'
        else:
            eta_text = ''
        print(
            f'{label} {event_id} [{index:03d}/{len(objects):03d}] {mode}; '
            f"strict={summary['primary_hybrid_status']}{eta_text}"
        )

    summary_frame = pd.DataFrame(summaries).sort_values('reference_time_unix').reset_index(drop=True)
    candidates_frame = pd.concat(candidate_frames, ignore_index=True)
    print(f'{label}: finished; {resumed} event(s) loaded from checkpoints.')
    print(f'{label}: elapsed {(time.perf_counter() - start_clock) / 60.0:.1f} min')
    return summary_frame, candidates_frame


## 2. Corrected strict reassociation


In [ ]:
actual_summary, actual_candidates = process_campaign(
    time_offset_sec=0,
    checkpoint_folder=ACTUAL_CHECKPOINTS,
    label='actual',
)

actual_summary.to_csv(OUT / 'strict_consolidated_associations.csv', index=False)
actual_candidates.to_csv(
    OUT / 'strict_catalog_candidates_top50.csv.gz', index=False, compression='gzip'
)

                                                                          
assigned = actual_summary[actual_summary['primary_hybrid_status'].eq('unique')].copy()
assert (assigned['predicted_elevation_deg'] > 0).all()
assert (assigned['beam_separation_deg'] <= BEAM_LIMIT_DEG).all()
assert (assigned['range_residual_km'].abs() < assigned['primary_range_gate_km']).all()
assert (
    assigned['range_rate_residual_km_s'].abs() < assigned['primary_rate_gate_km_s']
).all()

                                                                                     
                                                       
group_sizes = membership.groupby('consolidated_detection_id').size()
singleton_map = membership[
    membership['consolidated_detection_id'].isin(group_sizes[group_sizes.eq(1)].index)
].copy()
new_single = singleton_map.merge(
    actual_summary[['event_id', 'primary_hybrid_status', 'matched_NORAD_CAT_ID']],
    left_on='consolidated_detection_id', right_on='event_id', how='left', validate='one_to_one'
).merge(
    original_assoc[['object_detection_id', 'primary_hybrid_status', 'matched_NORAD_CAT_ID']],
    on='object_detection_id', how='left', validate='one_to_one', suffixes=('_new', '_old')
)
status_mismatch = new_single['primary_hybrid_status_new'].ne(
    new_single['primary_hybrid_status_old']
)
both_unique = (
    new_single['primary_hybrid_status_new'].eq('unique')
    & new_single['primary_hybrid_status_old'].eq('unique')
)
new_ids = pd.to_numeric(new_single['matched_NORAD_CAT_ID_new'], errors='coerce')
old_ids = pd.to_numeric(new_single['matched_NORAD_CAT_ID_old'], errors='coerce')
identity_mismatch = both_unique & new_ids.ne(old_ids)
regression_failures = new_single[status_mismatch | identity_mismatch]
if len(regression_failures):
    display(regression_failures)
    raise AssertionError(
        'Corrected reassociation failed the singleton regression test. '
        'Do not use these results.'
    )

strict_counts = actual_summary['primary_hybrid_status'].value_counts().reindex(
    ['unique', 'ambiguous', 'none'], fill_value=0
)
print('STRICT ASSOCIATION PASSED ALL AUDITS')
print('  unique:', int(strict_counts['unique']))
print('  ambiguous:', int(strict_counts['ambiguous']))
print('  unresolved:', int(strict_counts['none']))
print('  unique NORAD IDs:', int(assigned['matched_NORAD_CAT_ID'].nunique()))
print('  singleton regression mismatches:', len(regression_failures))


## 3. Empirical residual model and time-shift controls


In [ ]:
strict_training = actual_summary[
    actual_summary['primary_hybrid_status'].eq('unique')
].copy()
if len(strict_training) < 20:
    raise RuntimeError('Too few strict matches to estimate an empirical residual model.')

range_residuals = strict_training['range_residual_km'].to_numpy(float)
rate_residuals = strict_training['range_rate_residual_km_s'].to_numpy(float)
range_center = float(np.median(range_residuals))
rate_center = float(np.median(rate_residuals))
range_sigma = max(
    1.4826 * float(np.median(np.abs(range_residuals - range_center))),
    EMPIRICAL_RANGE_SIGMA_FLOOR_KM,
)
rate_sigma = max(
    1.4826 * float(np.median(np.abs(rate_residuals - rate_center))),
    EMPIRICAL_RATE_SIGMA_FLOOR_KM_S,
)
z_range = np.clip((range_residuals - range_center) / range_sigma, -4, 4)
z_rate = np.clip((rate_residuals - rate_center) / rate_sigma, -4, 4)
correlation = float(np.corrcoef(z_range, z_rate)[0, 1])
if not np.isfinite(correlation):
    correlation = 0.0
correlation = float(np.clip(correlation, -0.80, 0.80))
inverse_correlation = np.linalg.inv(np.array([[1.0, correlation], [correlation, 1.0]]))

residual_model = {
    'training_matches': int(len(strict_training)),
    'range_center_km': range_center,
    'rate_center_km_s': rate_center,
    'range_sigma_km': range_sigma,
    'rate_sigma_km_s': rate_sigma,
    'correlation': correlation,
    'chi_square_2df_99_threshold': EMPIRICAL_CHI2_99,
    'absolute_range_cap_km': EMPIRICAL_ABS_RANGE_CAP_KM,
    'absolute_rate_cap_km_s': EMPIRICAL_ABS_RATE_CAP_KM_S,
}
(OUT / 'empirical_residual_model.json').write_text(json.dumps(residual_model, indent=2))
print(json.dumps(residual_model, indent=2))


def add_empirical_distance(candidates):
    frame = candidates.copy()
    def as_boolean(series):
        if pd.api.types.is_bool_dtype(series):
            return series.fillna(False)
        return (
            series.astype(str).str.strip().str.lower()
            .map({'true': True, 'false': False, '1': True, '0': False})
            .fillna(False)
        )
    zr = (frame['range_residual_km'].to_numpy(float) - range_center) / range_sigma
    zv = (frame['range_rate_residual_km_s'].to_numpy(float) - rate_center) / rate_sigma
    z = np.column_stack([zr, zv])
    frame['empirical_d2'] = np.einsum('ij,jk,ik->i', z, inverse_correlation, z)
    above_horizon = as_boolean(frame['above_horizon'])
    within_beam = as_boolean(frame['within_beam_model'])
    frame['meets_empirical_gates'] = (
        above_horizon
        & within_beam
        & (frame['range_residual_km'].abs() < EMPIRICAL_ABS_RANGE_CAP_KM)
        & (frame['range_rate_residual_km_s'].abs() < EMPIRICAL_ABS_RATE_CAP_KM_S)
        & (frame['empirical_d2'] < EMPIRICAL_CHI2_99)
    )
    return frame


def empirical_event_summary(summary_frame, candidates_frame):
    candidates_scored = add_empirical_distance(candidates_frame)
    rows = []
    for event_id, group in candidates_scored.groupby('event_id', sort=False):
        above_horizon = group['above_horizon'].astype(str).str.lower().eq('true')
        within_beam = group['within_beam_model'].astype(str).str.lower().eq('true')
        if pd.api.types.is_bool_dtype(group['above_horizon']):
            above_horizon = group['above_horizon'].fillna(False)
        if pd.api.types.is_bool_dtype(group['within_beam_model']):
            within_beam = group['within_beam_model'].fillna(False)
        geometry_ranked = group[
            above_horizon & within_beam
        ].sort_values('empirical_d2')
        if geometry_ranked.empty:
                                                                               
                                                                               
                                                                             
                                                                       
            diagnostic = group.sort_values('empirical_d2').iloc[0]
            rows.append({
                'event_id': str(event_id),
                'empirical_status': 'none',
                'number_meeting_empirical': 0,
                'empirical_candidate_NORAD_CAT_ID': None,
                'empirical_candidate_OBJECT_NAME': None,
                'empirical_best_d2': float(diagnostic['empirical_d2']),
                'empirical_d2_margin': np.nan,
                'empirical_range_residual_km': float(diagnostic['range_residual_km']),
                'empirical_rate_residual_km_s': float(diagnostic['range_rate_residual_km_s']),
                'empirical_beam_separation_deg': float(diagnostic['beam_separation_deg']),
                'empirical_predicted_elevation_deg': float(diagnostic['predicted_elevation_deg']),
            })
            continue
        passing = geometry_ranked[geometry_ranked['meets_empirical_gates']]
        passing = passing.drop_duplicates('NORAD_CAT_ID', keep='first')
        status = criterion_status(len(passing))
        best = passing.iloc[0] if len(passing) else geometry_ranked.iloc[0]
        if len(geometry_ranked) > 1:
            d2_margin = float(geometry_ranked.iloc[1]['empirical_d2'] - geometry_ranked.iloc[0]['empirical_d2'])
        else:
            d2_margin = math.inf
        rows.append({
            'event_id': str(event_id),
            'empirical_status': status,
            'number_meeting_empirical': int(len(passing)),
            'empirical_candidate_NORAD_CAT_ID': int(best['NORAD_CAT_ID']) if status == 'unique' else None,
            'empirical_candidate_OBJECT_NAME': str(best.get('OBJECT_NAME', '')) if status == 'unique' else None,
            'empirical_best_d2': float(best['empirical_d2']),
            'empirical_d2_margin': d2_margin,
            'empirical_range_residual_km': float(best['range_residual_km']),
            'empirical_rate_residual_km_s': float(best['range_rate_residual_km_s']),
            'empirical_beam_separation_deg': float(best['beam_separation_deg']),
            'empirical_predicted_elevation_deg': float(best['predicted_elevation_deg']),
        })
    empirical = pd.DataFrame(rows)
    result = summary_frame.merge(empirical, on='event_id', how='left', validate='one_to_one')
    result['radar_quality_flag'] = (
        (result['duration_sec'] >= QUALITY_MIN_DURATION_SEC)
        | (result['peak_detector_score'] >= QUALITY_MIN_SCORE)
    )
    result['tier_b_shape_candidate'] = (
        result['primary_hybrid_status'].eq('none')
        & result['empirical_status'].eq('unique')
        & (result['empirical_d2_margin'] >= EMPIRICAL_MIN_D2_MARGIN)
        & result['radar_quality_flag']
    )
    return result, candidates_scored


actual_scored, actual_candidates_scored = empirical_event_summary(
    actual_summary, actual_candidates
)
observed_tier_b_shape = int(actual_scored['tier_b_shape_candidate'].sum())
print('Observed Tier-B-shaped candidates before negative-control calibration:', observed_tier_b_shape)


In [ ]:
control_rows = []

for shift_sec in TIME_SHIFT_CONTROLS_SEC:
    shift_label = f"shift_{'p' if shift_sec >= 0 else 'm'}{abs(int(shift_sec)):04d}s"
    shift_folder = SHIFT_CHECKPOINTS / shift_label
    shifted_summary, shifted_candidates = process_campaign(
        time_offset_sec=shift_sec,
        checkpoint_folder=shift_folder,
        label=shift_label,
    )
    shifted_scored, _ = empirical_event_summary(shifted_summary, shifted_candidates)
    strict_shift_counts = shifted_scored['primary_hybrid_status'].value_counts()
    empirical_shift_counts = shifted_scored['empirical_status'].value_counts()
    control_rows.append({
        'time_shift_sec': int(shift_sec),
        'strict_unique': int(strict_shift_counts.get('unique', 0)),
        'strict_ambiguous': int(strict_shift_counts.get('ambiguous', 0)),
        'empirical_unique': int(empirical_shift_counts.get('unique', 0)),
        'empirical_ambiguous': int(empirical_shift_counts.get('ambiguous', 0)),
        'tier_b_shape_candidates': int(shifted_scored['tier_b_shape_candidate'].sum()),
        'events': int(len(shifted_scored)),
    })

controls = pd.DataFrame(control_rows)
controls.to_csv(OUT / 'time_shift_false_association_controls.csv', index=False)
null_tier_b_mean = float(controls['tier_b_shape_candidates'].mean())
if observed_tier_b_shape > 0:
    estimated_tier_b_fdr = min(1.0, null_tier_b_mean / observed_tier_b_shape)
else:
    estimated_tier_b_fdr = 0.0
tier_b_control_pass = bool(
    observed_tier_b_shape > 0 and estimated_tier_b_fdr <= MAX_CONTROL_FDR
)

print('Time-shift controls:')
display(controls)
print('Observed Tier-B-shaped candidates:', observed_tier_b_shape)
print('Mean shifted-time Tier-B-shaped candidates:', round(null_tier_b_mean, 3))
print('Estimated Tier-B FDR:', round(estimated_tier_b_fdr, 4))
print('Tier B control pass:', tier_b_control_pass)


## 4. Freeze audited outputs


In [ ]:
final = actual_scored.copy()
final['association_tier'] = 'unresolved'
final.loc[final['primary_hybrid_status'].eq('ambiguous'), 'association_tier'] = 'ambiguous_strict'
final.loc[final['primary_hybrid_status'].eq('unique'), 'association_tier'] = 'tier_a_strict'
if tier_b_control_pass:
    final.loc[final['tier_b_shape_candidate'], 'association_tier'] = 'tier_b_empirical'

final['final_NORAD_CAT_ID'] = np.nan
final['final_OBJECT_NAME'] = ''
tier_a = final['association_tier'].eq('tier_a_strict')
tier_b = final['association_tier'].eq('tier_b_empirical')
final.loc[tier_a, 'final_NORAD_CAT_ID'] = pd.to_numeric(
    final.loc[tier_a, 'matched_NORAD_CAT_ID'], errors='coerce'
)
final.loc[tier_a, 'final_OBJECT_NAME'] = final.loc[tier_a, 'matched_OBJECT_NAME'].fillna('')
final.loc[tier_b, 'final_NORAD_CAT_ID'] = pd.to_numeric(
    final.loc[tier_b, 'empirical_candidate_NORAD_CAT_ID'], errors='coerce'
)
final.loc[tier_b, 'final_OBJECT_NAME'] = final.loc[tier_b, 'empirical_candidate_OBJECT_NAME'].fillna('')

                                                                        
assert final.loc[tier_a, 'matched_NORAD_CAT_ID'].notna().all()
assert (final.loc[tier_a, 'predicted_elevation_deg'] > 0).all()
assert (final.loc[tier_a, 'beam_separation_deg'] <= BEAM_LIMIT_DEG).all()
if tier_b.any():
    assert (final.loc[tier_b, 'empirical_predicted_elevation_deg'] > 0).all()
    assert (final.loc[tier_b, 'empirical_beam_separation_deg'] <= BEAM_LIMIT_DEG).all()
    assert (final.loc[tier_b, 'empirical_best_d2'] < EMPIRICAL_CHI2_99).all()
    assert (final.loc[tier_b, 'empirical_d2_margin'] >= EMPIRICAL_MIN_D2_MARGIN).all()

tier_counts = final['association_tier'].value_counts().reindex(
    ['tier_a_strict', 'tier_b_empirical', 'ambiguous_strict', 'unresolved'],
    fill_value=0,
)
unique_final_ids = int(pd.to_numeric(final['final_NORAD_CAT_ID'], errors='coerce').nunique())

final.to_csv(OUT / 'final_consolidated_catalog_association.csv', index=False)
actual_candidates_scored.to_csv(
    OUT / 'final_catalog_candidates_top50.csv.gz', index=False, compression='gzip'
)

unique_catalog = final[final['final_NORAD_CAT_ID'].notna()].sort_values(
    ['final_NORAD_CAT_ID', 'reference_time_unix']
).drop_duplicates('final_NORAD_CAT_ID')
unique_catalog.to_csv(OUT / 'final_unique_catalog_objects.csv', index=False)

high_priority_unresolved = final[
    final['association_tier'].isin(['unresolved', 'ambiguous_strict'])
    & final['radar_quality_flag']
].sort_values(['duration_sec', 'peak_detector_score'], ascending=False)
high_priority_unresolved.to_csv(OUT / 'high_priority_unresolved_events.csv', index=False)

tier_table = pd.DataFrame({
    'association_tier': tier_counts.index,
    'events': tier_counts.values.astype(int),
})
tier_table.to_csv(OUT / 'association_tier_summary.csv', index=False)

campaign_summary = {
    'valid': True,
    'config_hash': CONFIG_HASH,
    'frozen_detector': {
        'config_id': EXPECTED_CONFIG_ID,
        'method': EXPECTED_METHOD,
        'threshold': EXPECTED_THRESHOLD,
        'track_rule': EXPECTED_TRACK_RULE,
    },
    'radar_fragments': int(len(selected)),
    'consolidated_events': int(len(final)),
    'accepted_fragment_links': int(len(accepted_links)),
    'multi_fragment_events': int((objects['number_fragments'] > 1).sum()),
    'strict_primary': {
        'unique_associations': int(strict_counts['unique']),
        'ambiguous_associations': int(strict_counts['ambiguous']),
        'unresolved_events': int(strict_counts['none']),
        'unique_norad_ids': int(assigned['matched_NORAD_CAT_ID'].nunique()),
    },
    'final_tiers': {key: int(value) for key, value in tier_counts.items()},
    'final_unique_norad_ids': unique_final_ids,
    'tier_b_control': {
        'observed_shape_candidates': observed_tier_b_shape,
        'mean_shifted_shape_candidates': null_tier_b_mean,
        'estimated_fdr': estimated_tier_b_fdr,
        'maximum_allowed_fdr': MAX_CONTROL_FDR,
        'promoted': tier_b_control_pass,
        'time_shifts_sec': TIME_SHIFT_CONTROLS_SEC,
    },
    'audit': {
        'singleton_regression_events': int(len(new_single)),
        'singleton_regression_mismatches': int(len(regression_failures)),
        'all_tier_a_pass_range_rate_beam_horizon': True,
        'invalid_previous_119_association_result_used': False,
    },
    'scope_warning': (
        'All 24 temporal hours are included, but only the approximately '
        '1026.639-1084.050 km CFAR-valid slant-range window was searched.'
    ),
}
(OUT / 'final_catalog_campaign_summary.json').write_text(
    json.dumps(campaign_summary, indent=2)
)

print('FINAL AUDITED COUNTS')
display(tier_table)
print('Final unique NORAD IDs:', unique_final_ids)
print('High-priority unresolved events:', len(high_priority_unresolved))
print('Summary:', OUT / 'final_catalog_campaign_summary.json')


In [ ]:
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
plt.rcParams.update({'figure.dpi': 130, 'savefig.dpi': 300, 'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
colors = {'tier_a_strict': '#176B87', 'tier_b_empirical': '#3AA17E', 'ambiguous_strict': '#F2A541', 'unresolved': '#B7C4CF'}
fig, axes = plt.subplots(2, 2, figsize=(13.2, 8.8), constrained_layout=True)
ax = axes[0, 0]
bars = ax.bar(['Input\nfragments', 'Consolidated\nevents'], [len(selected), len(final)], color=['#B7C4CF', '#176B87'])
ax.set_ylabel('Count')
ax.set_title('(a) Radar-fragment consolidation', loc='left', fontweight='bold')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')
ax = axes[0, 1]
labels = ['Tier A\nstrict', 'Tier B\ncontrolled', 'Ambiguous', 'Unresolved']
keys = ['tier_a_strict', 'tier_b_empirical', 'ambiguous_strict', 'unresolved']
values = [int(tier_counts[key]) for key in keys]
bars = ax.bar(labels, values, color=[colors[key] for key in keys])
ax.set_ylabel('Consolidated radar events')
ax.set_title('(b) Audited catalog status', loc='left', fontweight='bold')
for bar in bars:
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), str(int(bar.get_height())), ha='center', va='bottom', fontweight='bold')
ax = axes[1, 0]
for status, color, label in [('none', colors['unresolved'], 'Unresolved'), ('ambiguous', colors['ambiguous_strict'], 'Ambiguous'), ('unique', colors['tier_a_strict'], 'Tier A strict')]:
    mask = actual_summary['primary_hybrid_status'].eq(status)
    ax.scatter(actual_summary.loc[mask, 'normalized_primary_range_residual'], actual_summary.loc[mask, 'normalized_primary_rate_residual'], s=28 if status == 'unique' else 18, alpha=0.78, color=color, label=label, edgecolor='none')
ax.add_patch(Rectangle((-1, -1), 2, 2, fill=False, lw=1.7, edgecolor='#8B1E3F', linestyle='--'))
ax.axhline(0, color='0.82', lw=0.8)
ax.axvline(0, color='0.82', lw=0.8)
ax.set_xlabel('Range residual / hybrid range gate')
ax.set_ylabel('Range-rate residual / hybrid rate gate')
ax.set_title('(c) Required range-rate gate audit', loc='left', fontweight='bold')
ax.legend(frameon=False)
ax = axes[1, 1]
x = np.arange(len(controls))
width = 0.36
ax.bar(x - width / 2, controls['strict_unique'], width, color='#176B87', label='Strict unique')
ax.bar(x + width / 2, controls['tier_b_shape_candidates'], width, color='#3AA17E', label='Tier-B-shaped')
ax.axhline(observed_tier_b_shape, color='#8B1E3F', linestyle='--', label='Observed Tier-B-shaped')
ax.set_xticks(x)
ax.set_xticklabels([f'{int(value / 60):+d} min' for value in controls['time_shift_sec']])
ax.set_ylabel('False association candidates')
ax.set_title('(d) Shifted-time negative controls', loc='left', fontweight='bold')
ax.legend(frameon=False)
PANEL_PNG = OUT / 'final_catalog_audit_2x2.png'
PANEL_PDF = OUT / 'final_catalog_audit_2x2.pdf'
fig.savefig(PANEL_PNG, bbox_inches='tight')
fig.savefig(PANEL_PDF, bbox_inches='tight')
plt.show()
readme = f"research final audited catalog association\n=====================================\n\nFrozen detector: {EXPECTED_CONFIG_ID}, threshold {EXPECTED_THRESHOLD}, rule {EXPECTED_TRACK_RULE}\nInput radar fragments: {len(selected)}\nConsolidated radar events: {len(final)}\nTier A strict associations: {int(tier_counts['tier_a_strict'])}\nTier B empirically controlled associations: {int(tier_counts['tier_b_empirical'])}\nAmbiguous strict associations: {int(tier_counts['ambiguous_strict'])}\nUnresolved radar events: {int(tier_counts['unresolved'])}\nFinal distinct NORAD IDs: {unique_final_ids}\n\nTier A requires:\n- propagation at the consolidated fit-center time using the latest-prior TLE;\n- |range residual| < max(1 km, 0.1% of observed slant range);\n- |range-rate residual| < max(20 m/s, 1% of absolute observed radial rate);\n- predicted elevation above zero and beam separation <= 5 degrees.\n\nTier B is reported separately. It uses a robust residual ellipse learned from\nTier A, absolute safety caps, a likelihood-separation requirement, a radar-quality\nflag, and four shifted-time negative controls. It is promoted only when the\nestimated campaign-level false-discovery rate is <= {MAX_CONTROL_FDR:.0%}.\n\nAudit result:\n- all 147 non-merged events reproduce the original valid catalog result;\n- every assigned Tier A identity passes range, rate, beam, and horizon gates;\n- the previous invalid 119-association output is not read or reused.\n\nScope:\nThese data span all 24 temporal hours but only the approximately\n1026.639-1084.050 km CFAR-valid slant-range window. Unresolved events are not\nautomatically uncataloged debris.\n"
(OUT / 'README_final_catalog.txt').write_text(readme)
print('FINISHED — FINAL AUDITED OUTPUTS SAVED')
print('Folder:', OUT)
print('Main table:', OUT / 'final_consolidated_catalog_association.csv')
print('Summary:', OUT / 'final_catalog_campaign_summary.json')
print('Panel:', PANEL_PNG)


## Success condition
